In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, RocCurveDisplay


In [ ]:
df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

In [ ]:
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = ["gender", "Partner", "Dependents", "PhoneService", 
                        "InternetService", "Contract", "PaymentMethod"]

X = df[numerical_features + categorical_features]
y = df["Churn"].map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2, random_state=42
)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(), categorical_features)
                                              ])

pipeline = Pipeline(steps=[
    ("preprocessor" , preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced"))
])

param_grid = {
    'classifier__C' : [0.01, 0.1, 1, 10, 100],
    'classifier__l1_ratio': [0, 1],
    'classifier__solver' : ['liblinear']                      
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

best_model = grid_search.best_estimator_


In [ ]:
RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing') # Baseline diagonal line
plt.title('ROC Curve (Entire 10-Column Model)')
plt.legend()
plt.show()

In [ ]:
print(classification_report(y_test, best_model.predict(X_test)))

In [ ]:
import joblib
joblib.dump(best_model, "../model/churn_model.pkl")

In [ ]:
print(df["Contract"].unique())
print(df["PaymentMethod"].unique())
print(df["InternetService"].unique())

In [ ]:
print(df["Churn"].value_counts())